# Renda domiciliar e orçamento para hardware

**Pergunta:** Que faixas de renda as famílias brasileiras ocupam, e a partir daí, para quem um produto caro (ex.: fogão inteligente) cabe no orçamento?

**Fonte:** PNADC Anual 2025 (Visita 1) — **renda domiciliar**, grão de **domicílio**.

**Por que domiciliar e não per capita:** quem paga o produto é o orçamento da casa inteira.

**Método:** faixas de **salário mínimo** sobre a renda domiciliar (transparente e ancorado no SM). Renda é contínua e **assimétrica** → olhamos a distribuição por faixa, não a "renda média" (que a cauda dos ricos puxa pra cima).

**Grão:** colapsamos para 1 linha por `id_domicilio` (= UPA + V1008 + V1014) e usamos o peso do domicílio — depois de *verificar* que o peso é constante dentro do domicílio.

In [ ]:
import sys
from pathlib import Path

import duckdb

root = Path.cwd()
while not (root / "data").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

GLOB = str(root / "data" / "pnadc" / "2025" / "mapped" / "*.parquet")
con = duckdb.connect()

# --- knobs (ajuste conforme a realidade) ---
SM_2025 = 1518.0       # salario minimo 2025 (R$)
PRECO_FOGAO = 3000.0   # preco hipotetico do fogao inteligente (R$)

ID = ("CAST(upa AS VARCHAR) || '_' || CAST(numero_de_selecao_do_domicilio AS VARCHAR)"
      " || '_' || CAST(painel AS VARCHAR)")
RENDA = "rendimento_habitual_domiciliar_mensal"


def faixa_sql(col="renda"):
    return f"""CASE
        WHEN {col} IS NULL      THEN '0) sem informacao'
        WHEN {col} < 1*{SM_2025}  THEN '1) ate 1 SM'
        WHEN {col} < 2*{SM_2025}  THEN '2) 1-2 SM'
        WHEN {col} < 3*{SM_2025}  THEN '3) 2-3 SM'
        WHEN {col} < 5*{SM_2025}  THEN '4) 3-5 SM'
        WHEN {col} < 10*{SM_2025} THEN '5) 5-10 SM'
        ELSE '6) 10+ SM'
    END"""

## 1. Verificação: o peso é constante dentro do domicílio?

In [ ]:
bad = con.execute(f"""
    SELECT COUNT(*) FROM (
        SELECT {ID} AS id FROM read_parquet('{GLOB}')
        GROUP BY id HAVING COUNT(DISTINCT peso_com_calibracao) > 1
    )
""").fetchone()[0]
print(f"Domicilios com peso divergente entre moradores: {bad}")
print("OK: peso constante -> serve como peso do domicilio" if bad == 0
      else "ATENCAO: peso varia no domicilio, rever metodo")

## 2. Colapsa para grão de domicílio + sanity check

In [ ]:
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW domicilios AS
    SELECT {ID} AS id_domicilio,
           any_value({RENDA}) AS renda,
           any_value(peso_com_calibracao) AS peso
    FROM read_parquet('{GLOB}')
    GROUP BY 1
""")
r = con.execute(f"""
    SELECT CAST(SUM(peso) AS BIGINT) AS domicilios_estimados,
           (SELECT CAST(SUM(peso_com_calibracao) AS BIGINT) FROM read_parquet('{GLOB}')) AS pessoas_estimadas
    FROM domicilios
""").df()
print(r.to_string(index=False))
print("\n(o MESMO peso somado por domicilio da' ~domicilios; somado por pessoa da' ~populacao)")

## 3. Distribuição dos domicílios por faixa de renda

In [ ]:
faixas = con.execute(f"""
    WITH c AS (SELECT peso, {faixa_sql()} AS faixa FROM domicilios)
    SELECT faixa,
           CAST(SUM(peso) AS BIGINT) AS domicilios,
           ROUND(100 * SUM(peso) / SUM(SUM(peso)) OVER (), 1) AS pct
    FROM c GROUP BY faixa ORDER BY faixa
""").df()
faixas["pct_acumulado"] = faixas["pct"].cumsum().round(1)
print(faixas.to_string(index=False))

## 4. Acessibilidade: o produto cabe no orçamento de cada faixa?

Para cada faixa, o preço do produto como **% de um mês de renda domiciliar** (renda média ponderada da faixa). Quanto menor o %, mais viável.

In [ ]:
aff = con.execute(f"""
    WITH c AS (SELECT peso, renda, {faixa_sql()} AS faixa
               FROM domicilios WHERE renda IS NOT NULL)
    SELECT faixa,
           CAST(ROUND(SUM(renda * peso) / SUM(peso)) AS BIGINT) AS renda_media_mensal,
           ROUND(100 * {PRECO_FOGAO} / (SUM(renda * peso) / SUM(peso)), 0) AS preco_pct_de_1_mes
    FROM c GROUP BY faixa ORDER BY faixa
""").df()
print(f"Fogao inteligente hipotetico: R$ {PRECO_FOGAO:,.0f}\n")
print(aff.to_string(index=False))

## Caveats e próximos passos
- `PRECO_FOGAO` e `SM_2025` são **knobs** — trocar pelo preço real do produto e o SM vigente.
- Acessibilidade aqui é o preço sobre **um mês** de renda; uma compra única costuma ser avaliada sobre alguns meses / capacidade de poupança — dá pra refinar.
- Renda assimétrica: usamos média **por faixa** (faixas estreitas), mas a média global enganaria — a distribuição por faixa é a leitura honesta.
- Comparar com 2010 (deflacionado) mostraria a evolução do poder de compra.